In [3]:
import csv
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
PASTA_DADOS = Path(r'C:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\analises_2025')

df_umdia = pd.read_excel(PASTA_DADOS / 'table_oneday.xlsx')
df_umahora = pd.read_excel(PASTA_DADOS / 'table_onehour.xlsx')
df_umminuto = pd.read_excel(PASTA_DADOS / 'table_oneminute.xlsx')
df_dezminutos = pd.read_excel(PASTA_DADOS / 'table_tenminutes.xlsx')

## Reconstrução da tabela horária a partir dos dados por minuto

Esta etapa preenche apenas valores ausentes da `table_onehour.xlsx`. Para cada hora, são usados os dados da `table_oneminute.xlsx` somente quando houver pelo menos 45 registros válidos. Os valores horários originais não são substituídos.

In [4]:
from pathlib import Path

MIN_OBSERVACOES_POR_HORA = 45
COLUNAS_SOMA = ['Pluviometro_Tot']  # Precipitação é acumulada na hora; as demais variáveis usam média.
FATORES_CONVERSAO_MINUTO = {
    'Piranometro_G_Avg': 0.1, 'Piranometro_G_Max': 0.1, 'Piranometro_G_Min': 0.1,
    'Piranometro_R_Avg': 0.1, 'Piranometro_R_Max': 0.1, 'Piranometro_R_Min': 0.1,
    'Vento_Vel_Avg': 0.01, 'Vento_Vel_Max': 0.01, 'Vento_Vel_Min': 0.01,
}
ARQUIVO_SAIDA = PASTA_DADOS / 'table_onehour_completada.xlsx'

def recuperar_linhas_csv_embutidas(dados):
    """Recupera registros que foram armazenados inteiros na coluna TIMESTAMP."""
    dados = dados.copy().astype('object')
    texto_timestamp = dados['TIMESTAMP'].astype(str)
    mascara = texto_timestamp.str.contains(',', regex=False)
    if mascara.any():
        linhas = [next(csv.reader([linha])) for linha in texto_timestamp[mascara]]
        recuperadas = pd.DataFrame(linhas, index=dados.index[mascara], columns=dados.columns)
        dados.loc[mascara, dados.columns] = recuperadas
        print(f'{mascara.sum()} linhas por minuto recuperadas da coluna TIMESTAMP.')
    return dados

hora_original = df_umahora.copy()
minuto_original = recuperar_linhas_csv_embutidas(df_umminuto)

for base in (hora_original, minuto_original):
    base['TIMESTAMP'] = pd.to_datetime(base['TIMESTAMP'], errors='coerce')

hora_original = hora_original.dropna(subset=['TIMESTAMP']).sort_values('TIMESTAMP')
minuto_original = minuto_original.dropna(subset=['TIMESTAMP']).sort_values('TIMESTAMP')

colunas_candidatas = sorted(
    set(hora_original.columns).intersection(minuto_original.columns) - {'TIMESTAMP', 'RECORD'}
)
colunas_comuns = [
    coluna for coluna in colunas_candidatas
    if pd.to_numeric(hora_original[coluna], errors='coerce').notna().any()
    and pd.to_numeric(minuto_original[coluna], errors='coerce').notna().any()
]
if not colunas_comuns:
    raise ValueError('Nenhuma coluna numérica com o mesmo nome foi encontrada nas tabelas horária e por minuto.')

for coluna in colunas_comuns:
    hora_original[coluna] = pd.to_numeric(hora_original[coluna], errors='coerce')
    minuto_original[coluna] = pd.to_numeric(minuto_original[coluna], errors='coerce')
    minuto_original[coluna] *= FATORES_CONVERSAO_MINUTO.get(coluna, 1)

minuto_original['hora'] = minuto_original['TIMESTAMP'].dt.floor('h')
agregacoes = {
    coluna: ('sum' if coluna in COLUNAS_SOMA else 'mean')
    for coluna in colunas_comuns
}
hora_minuto = minuto_original.groupby('hora').agg(agregacoes)
contagens_validas = minuto_original.groupby('hora')[colunas_comuns].count()
hora_minuto['n_obs_minuto'] = minuto_original.groupby('hora')['TIMESTAMP'].count()
for coluna in colunas_comuns:
    hora_minuto.loc[contagens_validas[coluna] < MIN_OBSERVACOES_POR_HORA, coluna] = pd.NA
hora_minuto = hora_minuto.reset_index().rename(columns={'hora': 'TIMESTAMP'})

inicio = min(hora_original['TIMESTAMP'].min(), hora_minuto['TIMESTAMP'].min()).floor('h')
fim = max(hora_original['TIMESTAMP'].max(), hora_minuto['TIMESTAMP'].max()).floor('h')
grade_horaria = pd.DataFrame({'TIMESTAMP': pd.date_range(inicio, fim, freq='h')})

hora_unica = hora_original.drop_duplicates('TIMESTAMP', keep='first')
completada = grade_horaria.merge(hora_unica, on='TIMESTAMP', how='left')
completada = completada.merge(hora_minuto, on='TIMESTAMP', how='left', suffixes=('', '_minuto'))

originais_preenchidos = completada[colunas_comuns].notna().any(axis=1)
preenchimentos_por_coluna = {}
foi_preenchido = pd.Series(False, index=completada.index)
for coluna in colunas_comuns:
    coluna_minuto = f'{coluna}_minuto'
    faltavam = completada[coluna].isna()
    disponivel_no_minuto = completada[coluna_minuto].notna()
    preencher = faltavam & disponivel_no_minuto
    preenchimentos_por_coluna[coluna] = int(preencher.sum())
    completada.loc[preencher, coluna] = completada.loc[preencher, coluna_minuto]
    foi_preenchido |= preencher

completada['origem_registro'] = 'table_onehour'
completada.loc[~originais_preenchidos & foi_preenchido, 'origem_registro'] = 'table_oneminute'
completada.loc[originais_preenchidos & foi_preenchido, 'origem_registro'] = 'mista'
completada.loc[~originais_preenchidos & ~foi_preenchido, 'origem_registro'] = 'sem_dados_suficientes'

colunas_auxiliares = [f'{coluna}_minuto' for coluna in colunas_comuns]
completada = completada.drop(columns=colunas_auxiliares)
completada.to_excel(ARQUIVO_SAIDA, index=False)

print(f'Arquivo criado: {ARQUIVO_SAIDA.resolve()}')
print(f'Cobertura mínima exigida: {MIN_OBSERVACOES_POR_HORA} registros por hora')
print('Preenchimentos por coluna:')
display(pd.Series(preenchimentos_por_coluna, name='valores_preenchidos').to_frame())
print('Origem dos registros:')
display(completada['origem_registro'].value_counts().to_frame('quantidade'))
print('Colunas comuns utilizadas:')
print(colunas_comuns)
print('Colunas da tabela horária sem correspondência na tabela por minuto:')
print(sorted(set(hora_original.columns) - set(minuto_original.columns) - {'TIMESTAMP'}))

37 linhas por minuto recuperadas da coluna TIMESTAMP.
Arquivo criado: C:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\analises_2025\table_onehour_completada.xlsx
Cobertura mínima exigida: 45 registros por hora
Preenchimentos por coluna:


,valores_preenchidos
Piranometro_G_Avg,3
Piranometro_G_Max,3
Piranometro_G_Min,3
Piranometro_R_Avg,1
Piranometro_R_Max,1
Piranometro_R_Min,1
Pluviometro_Tot,1
Vento_Vel_Avg,1
Vento_Vel_Max,1
Vento_Vel_Min,1


Origem dos registros:


,quantidade
origem_registro,
table_onehour,4641
sem_dados_suficientes,2056
mista,2
table_oneminute,1


Colunas comuns utilizadas:
['Piranometro_G_Avg', 'Piranometro_G_Max', 'Piranometro_G_Min', 'Piranometro_R_Avg', 'Piranometro_R_Max', 'Piranometro_R_Min', 'Pluviometro_Tot', 'Vento_Vel_Avg', 'Vento_Vel_Max', 'Vento_Vel_Min']
Colunas da tabela horária sem correspondência na tabela por minuto:
['Barometro_Avg', 'Barometro_Max', 'Barometro_Min', 'Temperatura_Avg', 'Temperatura_Max', 'Temperatura_Min', 'Umidade_Avg', 'Umidade_Max', 'Umidade_Min']


In [ ]:
df_umdia['Temperatura_Avg'] = pd.to_numeric(df_umdia['Temperatura_Avg'], errors='coerce') / 100

In [ ]:
df_umahora['Vento_Vel_Avg'] = pd.to_numeric(df_umdia['Vento_Vel_Avg'], errors='coerce') / 100

In [ ]:
print("Base de dados que registra os dados por dia")
df_umdia.isnull().sum()

In [ ]:
print("Base de dados que registra os dados por hora")
df_umahora.isnull().sum()

In [ ]:
print("Base de dados que registra os dados a cada dez minutos")
df_dezminutos.isnull().sum()

In [ ]:
print("Base de dados que registra os dados a cada minuto")
df_umminuto.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt

for df in [df_umdia, df_umahora, df_dezminutos]:
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df.dropna(subset=['TIMESTAMP'], inplace=True)

bases = [
    (df_umdia, 'Temperatura_Avg', 'Um dia'),
    (df_umahora, 'Temperatura_Avg', 'Uma hora'),
    (df_dezminutos, 'Temperatura (C Smp)', 'Dez minutos')
]

fig, axs = plt.subplots(3, 1, figsize=(14, 8), sharey=True)
axs = axs.flatten()

for i, (df, col_temp, titulo) in enumerate(bases):
    axs[i].plot(df['TIMESTAMP'], df[col_temp])
    axs[i].set_title(f'Temperatura ({titulo})')
    axs[i].set_xlabel("Tempo")
    axs[i].set_ylabel("Temperatura (°C)")
    axs[i].grid(True)

plt.tight_layout()
plt.suptitle("Variação da Temperatura em Diferentes Escalas de Tempo", fontsize=16, y=1.02)
plt.show()


In [ ]:
%pip install seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

variables = ['Temperatura_Avg', 'Piranometro_R_Avg', 'Vento_Vel_Avg', 'Barometro_Avg', 'Umidade_Avg']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2 linhas, 3 colunas
axes = axes.flatten()

for i, var in enumerate(variables):
    sns.boxplot(data=df_umdia, y=var, ax=axes[i])
    axes[i].set_title(f'Distribuição da variável {var}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Remove o último subplot vazio (posição 5)
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
variables = ['Temperatura_Avg', 'Piranometro_R_Avg', 'Vento_Vel_Avg', 'Barometro_Avg', 'Umidade_Avg']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2 linhas, 3 colunas
axes = axes.flatten()

for i, var in enumerate(variables):
    sns.boxplot(data=df_umahora, y=var, ax=axes[i])
    axes[i].set_title(f'Distribuição da variável {var}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Remove o último subplot vazio (posição 5)
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
variables = ['Temperatura (C Smp)', 'Piranometro_R_Avg (W/m² Avg)', 'Umidade (% Smp)', 'Vento_Vel_Avg (kilometers/hour Avg)', 'Barometro (hPa Smp)']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2 linhas, 3 colunas
axes = axes.flatten()

for i, var in enumerate(variables):
    sns.boxplot(data=df_dezminutos, y=var, ax=axes[i])
    axes[i].set_title(f'Distribuição da variável {var}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Remove o último subplot vazio (posição 5)
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

for df in [df_umdia, df_umahora, df_dezminutos]:
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df.dropna(subset=['TIMESTAMP'], inplace=True)

bases = [
    (df_umdia, 'Piranometro_R_Avg', 'Um dia'),
    (df_umahora, 'Piranometro_R_Avg', 'Uma hora'),
    (df_dezminutos, 'Piranometro_R_Avg (W/m² Avg)', 'Dez minutos')
]

fig = make_subplots(
    rows=3, cols=1,
    shared_yaxes=True,
    vertical_spacing=0.1,
    subplot_titles=[f'Radiação de onda longa ({titulo})' for _, _, titulo in bases]
)

for i, (df, col_temp, titulo) in enumerate(bases, start=1):
    fig.add_trace(
        go.Scatter(
            x=df['TIMESTAMP'],
            y=df[col_temp],
            mode='lines',
            name=f'Radiação de onda longa ({titulo})'
        ),
        row=i, col=1
    )

fig.update_layout(
    height=800,
    width=900,
    title_text="Variação da Radiação de onda longa em Diferentes Escalas de Tempo",
    showlegend=False
)

fig.update_xaxes(title_text="Tempo", row=3, col=1)
fig.update_yaxes(title_text="Radiação de onda longa (W/m²)", row=2, col=1)
fig.show()


In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

for df in [df_umdia, df_umahora, df_dezminutos]:
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df.dropna(subset=['TIMESTAMP'], inplace=True)

bases = [
    (df_umdia, 'Umidade_Avg', 'Um dia'),
    (df_umahora, 'Umidade_Avg', 'Uma hora'),
    (df_dezminutos, 'Umidade (% Smp)', 'Dez minutos')
]

fig = make_subplots(
    rows=3, cols=1,
    shared_yaxes=True,
    vertical_spacing=0.1,
    subplot_titles=[f'Umidade ({titulo})' for _, _, titulo in bases]
)

for i, (df, col_temp, titulo) in enumerate(bases, start=1):
    fig.add_trace(
        go.Scatter(
            x=df['TIMESTAMP'],
            y=df[col_temp],
            mode='lines',
            name=f'Umidade ({titulo})'
        ),
        row=i, col=1
    )

fig.update_layout(
    height=800,
    width=900,
    title_text="Variação da Umidade em Diferentes Escalas de Tempo",
    showlegend=False
)

fig.update_xaxes(title_text="Tempo", row=3, col=1)
fig.update_yaxes(title_text="Umidade (%)", row=2, col=1)
fig.show()


In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

for df in [df_umdia, df_umahora, df_dezminutos]:
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df.dropna(subset=['TIMESTAMP'], inplace=True)

bases = [
    (df_umdia, 'Vento_Vel_Avg', 'Um dia'),
    (df_umahora, 'Vento_Vel_Avg', 'Uma hora'),
    (df_dezminutos, 'Vento_Vel_Avg (kilometers/hour Avg)', 'Dez minutos')
]

fig = make_subplots(
    rows=3, cols=1,
    shared_yaxes=True,
    vertical_spacing=0.1,
    subplot_titles=[f'Velocidade do Vento ({titulo})' for _, _, titulo in bases]
)

for i, (df, col_temp, titulo) in enumerate(bases, start=1):
    fig.add_trace(
        go.Scatter(
            x=df['TIMESTAMP'],
            y=df[col_temp],
            mode='lines',
            name=f'Velocidade do tempo ({titulo})'
        ),
        row=i, col=1
    )

fig.update_layout(
    height=800,
    width=900,
    title_text="Variação da Velocidade do Vento em Diferentes Escalas de Tempo",
    showlegend=False
)

fig.update_xaxes(title_text="Tempo", row=3, col=1)
fig.update_yaxes(title_text="Velocidade do Vento (km/h Avg)", row=2, col=1)
fig.show()


In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

for df in [df_umdia, df_umahora, df_dezminutos]:
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df.dropna(subset=['TIMESTAMP'], inplace=True)

bases = [
    (df_umdia, 'Barometro_Avg', 'Um dia'),
    (df_umahora, 'Barometro_Avg', 'Uma hora'),
    (df_dezminutos, 'Barometro (hPa Smp)', 'Dez minutos')
]

fig = make_subplots(
    rows=3, cols=1,
    shared_yaxes=True,
    vertical_spacing=0.1,
    subplot_titles=[f'Pressão atmosférica ({titulo})' for _, _, titulo in bases]
)

for i, (df, col_temp, titulo) in enumerate(bases, start=1):
    fig.add_trace(
        go.Scatter(
            x=df['TIMESTAMP'],
            y=df[col_temp],
            mode='lines',
            name=f'Pressão atmosférica  ({titulo})'
        ),
        row=i, col=1
    )

fig.update_layout(
    height=800,
    width=900,
    title_text="Variação da Pressão Atmosférica em Diferentes Escalas de Tempo",
    showlegend=False
)

fig.update_xaxes(title_text="Tempo", row=3, col=1)
fig.update_yaxes(title_text="Pressão (hPa)", row=2, col=1)
fig.show()
